# Ghostexec + Unsloth / TRL (post-training + GRPO)

**Hardware:** use a **GPU** runtime (e.g. Colab T4 or better). CPU-only will not load 4-bit models.

This notebook is a **minimal** Colab-oriented stack:

1. **Optional SFT** — teach a small LM to emit **one JSON object per turn** matching `GhostexecAction` (see `models.py` in this repo).
2. **GRPO** — [Group Relative Policy Optimization](https://unsloth.ai/blog/grpo) with a **custom reward** that runs the real **OpenEnv / Ghostexec** simulator: parse JSON → `GhostexecEnvironment.reset()` → `step()` → scalar reward.

**Docs (RL concepts, GRPO, RLHF):** [Unsloth Reinforcement Learning guide](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide)

**Reference layout (OpenEnv + RL in Colab):** [Unsloth OpenEnv 2048 / gpt-oss GRPO notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/OpenEnv_gpt_oss_(20B)_Reinforcement_Learning_2048_Game.ipynb)

**Scope note:** Each reward evaluation uses a **fresh** env from the same deterministic scenario (`phase2_core.json` by default), so GRPO compares multiple **first-step** completions for the **same** initial briefing. Extending to full multi-step episodes is possible with a session client or state snapshots (future work).

**Run All:** set `GITHUB_REPO_URL` below if the notebook is not already inside the repo; use a **public** Unsloth model ID unless you set `HF_TOKEN` for gated models.


In [ ]:
# --- knobs (tune for real runs; keep small for "Run All" smoke) ---
import os

GITHUB_REPO_URL = os.environ.get("GHOSTEXEC_REPO_URL", "https://github.com/YOUR_ORG/YOUR_REPO.git").strip()

RUN_SFT = os.environ.get("GHOSTEXEC_RUN_SFT", "1") != "0"
SFT_SAMPLES = int(os.environ.get("GHOSTEXEC_SFT_SAMPLES", "128"))
SFT_MAX_STEPS = int(os.environ.get("GHOSTEXEC_SFT_MAX_STEPS", "1"))
GRPO_DATASET_ROWS = int(os.environ.get("GHOSTEXEC_GRPO_ROWS", "24"))
GRPO_MAX_STEPS = int(os.environ.get("GHOSTEXEC_GRPO_MAX_STEPS", "2"))
NUM_GENERATIONS = int(os.environ.get("GHOSTEXEC_NUM_GENERATIONS", "4"))

MODEL_NAME = os.environ.get(
    "GHOSTEXEC_MODEL",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
)
MAX_SEQ_LENGTH = int(os.environ.get("GHOSTEXEC_MAX_SEQ", "2048"))


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "packaging", "ninja", "wheel", "setuptools"],
    check=False,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "unsloth",
        "trl",
        "transformers",
        "accelerate",
        "bitsandbytes",
        "peft",
        "datasets",
        "matplotlib",
    ],
    check=False,
)
print("pip installs finished (unsloth + trl stack).")


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd() / "ghostexec", Path("/content/ghostexec")]
root = None
for p in candidates:
    if (p / "pyproject.toml").exists() and (p / "models.py").exists():
        root = p
        break

if root is None and GITHUB_REPO_URL and "YOUR_ORG" not in GITHUB_REPO_URL:
    target = Path("/content/ghostexec")
    if not (target / "pyproject.toml").exists():
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(target)], check=False)
    root = target if (target / "pyproject.toml").exists() else None

if root is None:
    raise RuntimeError(
        "Could not find ghostexec repo root. Upload the repo or set GHOSTEXEC_REPO_URL to a public git URL."
    )

os.chdir(root)
sys.path.insert(0, str(root))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=str(root), check=True)
print("Working directory:", root)


## Optional post-training (SFT): briefing → JSON action

Synthetic pairs from the scripted `smart_action` policy (`training/train.py`) so the model learns valid `GhostexecAction` JSON before RL.


In [ ]:
import json
import random
from pathlib import Path

from ghostexec.server.ghostexec_environment import GhostexecEnvironment
from training.train import smart_action

ROOT = Path.cwd()
scenario = ROOT / "scenarios" / "phase2_core.json"
env = GhostexecEnvironment(scenario)

INSTRUCTION = (
    "You are an executive assistant agent. Reply with EXACTLY one JSON object (no markdown fences) "
    "with keys matching GhostexecAction: action_type (string), and optional email_id, message_body, "
    "meeting_id, new_time, reason, task_id, contact_name, message."
)


def build_sft_rows(n: int, seed: int = 0) -> list[dict]:
    rng = random.Random(seed)
    rows: list[dict] = []
    for _ in range(n):
        obs = env.reset()
        act = smart_action(obs, rng)
        user = INSTRUCTION + "\n\n=== BRIEFING ===\n" + (obs.echoed_message or "")
        assistant = json.dumps(act.model_dump(mode="json"), ensure_ascii=False)
        rows.append({"user": user, "assistant": assistant})
    return rows


sft_rows = build_sft_rows(min(SFT_SAMPLES, 512), seed=42)
sft_path = ROOT / "outputs" / "training" / "colab_sft_messages.jsonl"
sft_path.parent.mkdir(parents=True, exist_ok=True)
with sft_path.open("w", encoding="utf-8") as fh:
    for r in sft_rows:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
print("Wrote", sft_path, "rows", len(sft_rows))


### SFT train (Unsloth + TRL `SFTTrainer`)

Skips if `RUN_SFT` is false. Uses 4-bit LoRA when available.


In [ ]:
import torch
from datasets import Dataset

model = None
tokenizer = None

if RUN_SFT:
    from unsloth import FastLanguageModel
    from trl import SFTConfig, SFTTrainer

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

    def to_text(example):
        msgs = [
            {"role": "user", "content": example["user"]},
            {"role": "assistant", "content": example["assistant"]},
        ]
        return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

    ds = Dataset.from_list(sft_rows)
    ds = ds.map(to_text, remove_columns=["user", "assistant"])

    _cuda = torch.cuda.is_available()
    _bf16 = _cuda and torch.cuda.is_bf16_supported()
    sft_args = SFTConfig(
        output_dir=str(ROOT / "outputs" / "training" / "unsloth_sft_ckpt"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        warmup_steps=1,
        max_steps=SFT_MAX_STEPS,
        logging_steps=1,
        learning_rate=2e-4,
        fp16=_cuda and not _bf16,
        bf16=_bf16,
        dataset_text_field="text",
        report_to="none",
    )
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=ds, args=sft_args)
    trainer.train()
    print("SFT done.")
else:
    print("Skipping SFT (RUN_SFT=0).")


## GRPO with Ghostexec reward

Reward function: `training.grpo_ghostexec_reward.ghostexec_env_step_reward` (fresh env, parse JSON, one `step()`).

Dataset: repeated **prompt** = instruction + first `reset()` briefing (deterministic).


In [ ]:
from datasets import Dataset
from ghostexec.server.ghostexec_environment import GhostexecEnvironment

scenario = ROOT / "scenarios" / "phase2_core.json"
_env = GhostexecEnvironment(scenario)
_brief = _env.reset().echoed_message or ""
GRPO_PROMPT = INSTRUCTION + "\n\n=== BRIEFING ===\n" + _brief

prompts = [GRPO_PROMPT] * GRPO_DATASET_ROWS
grpo_ds = Dataset.from_dict({"prompt": prompts})
print("GRPO dataset rows:", len(grpo_ds))


In [ ]:
import torch
from trl import GRPOConfig, GRPOTrainer

from training.grpo_ghostexec_reward import ghostexec_env_step_reward

if model is None or tokenizer is None:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

_cuda = torch.cuda.is_available()
_bf16 = _cuda and torch.cuda.is_bf16_supported()
grpo_args = GRPOConfig(
    output_dir=str(ROOT / "outputs" / "training" / "unsloth_grpo_ckpt"),
    per_device_train_batch_size=1,
    num_generations=NUM_GENERATIONS,
    max_steps=GRPO_MAX_STEPS,
    logging_steps=1,
    learning_rate=5e-6,
    fp16=_cuda and not _bf16,
    bf16=_bf16,
    report_to="none",
    remove_unused_columns=False,
)

trainer_grpo = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=ghostexec_env_step_reward,
    args=grpo_args,
    train_dataset=grpo_ds,
)
trainer_grpo.train()
print("GRPO finished.")


In [ ]:
import matplotlib.pyplot as plt

# TRL logs reward metrics when available
try:
    log_hist = trainer_grpo.state.log_history
    rewards = [h.get("rewards/ghostexec_env_step_reward/mean") for h in log_hist if "rewards/ghostexec_env_step_reward/mean" in h]
    if rewards:
        plt.figure(figsize=(8, 3))
        plt.plot(rewards, marker="o")
        plt.xlabel("log step")
        plt.ylabel("mean reward (Ghostexec first step)")
        plt.title("GRPO training (Ghostexec)")
        plt.grid(True, alpha=0.3)
        plt.show()
    else:
        print("No reward keys in log_history; printed keys:", [k for h in log_hist for k in h.keys()][:20])
except Exception as e:
    print("Plot skip:", e)
